In [3]:
import os
from dotenv import load_dotenv

from typing import TypedDict

from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END

from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()

llm = ChatGroq(
    model=os.environ.get("GROQ_MODEL_OPEN"),
    temperature=0,
)

In [5]:
class TranslationState(TypedDict):
    word: str
    meaning: str
    hindi: str
    gujarati: str

In [6]:
def analyze_word(state: TranslationState):

    word = state["word"]

    prompt = f"""
        You are a language expert.
        
        Analyze the English word:
        
        "{word}"
        
        Identify its most common meaning in everyday usage.
        
        Return only a short explanation of the meaning,
        in no more than 30 words.
        """

    response = llm.invoke(prompt)

    return {
        "meaning": response.content
    }

In [7]:
test_state = {
    "word": "bank",
    "meaning": "",
    "hindi": "",
    "gujarati": ""
}

result = analyze_word(test_state)

print(result)

{'meaning': 'A financial institution that accepts deposits, offers loans, and provides various money‑handling services.'}


In [8]:
builder = StateGraph(TranslationState)

builder.add_node("analyze_word", analyze_word)

builder.add_edge(START, "analyze_word")
builder.add_edge("analyze_word", END)

graph = builder.compile()

In [9]:
result = graph.invoke({
    "word": "bank",
    "meaning": "",
    "hindi": "",
    "gujarati": ""
})

print(result)

{'word': 'bank', 'meaning': 'A financial institution that accepts deposits, offers loans, and provides various money‑handling services.', 'hindi': '', 'gujarati': ''}


In [10]:
def translate_to_hindi(state: TranslationState):

    word = state["word"]
    meaning = state["meaning"]

    prompt = f"""
        Translate the English word "{word}" into Hindi.
        
        Context/meaning:
        {meaning}
        
        Return only the Hindi translation.
        Do not provide explanations.
        """

    response = llm.invoke(prompt)

    return {
        "hindi": response.content.strip()
    }

In [11]:
def translate_to_gujarati(state: TranslationState):

    word = state["word"]
    meaning = state["meaning"]

    prompt = f"""
        Translate the English word "{word}" into Gujarati.
        
        Context/meaning:
        {meaning}
        
        Return only the Gujarati translation.
        Do not provide explanations.
        """

    response = llm.invoke(prompt)

    return {
        "gujarati": response.content.strip()
    }

In [13]:
test_state = {
    "word": "bank",
    "meaning": "A financial institution where people keep and manage money.",
    "hindi": "",
    "gujarati": ""
}

result = translate_to_hindi(test_state)

print(result)


result = translate_to_gujarati(test_state)

print(result)

{'hindi': 'बैंक'}
{'gujarati': 'બેંક'}


In [20]:
builder = StateGraph(TranslationState)

builder.add_node("analyze_word", analyze_word)
builder.add_node("translate_to_hindi", translate_to_hindi)
builder.add_node("translate_to_gujarati", translate_to_gujarati)

builder.add_edge(START, "analyze_word")
builder.add_edge("analyze_word", "translate_to_hindi")
builder.add_edge("translate_to_hindi", "translate_to_gujarati")
builder.add_edge("translate_to_gujarati", END)

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["translate_to_hindi"]
)

In [24]:
user_input = input("Enter an English word: ")

config = {
    "configurable": {
        "thread_id": "translation-003"
    }
}

result = graph.invoke(
    {
    "word": user_input,
    "meaning": "",
    "hindi": "",
    "gujarati": ""
    },
    config=config
)

print("\nEnglish:", result["word"])
print("Meaning:", result["meaning"])
print("Hindi:", result["hindi"])
print("Gujarati:", result["gujarati"])
print("\n\nSaved State:")
for state in graph.get_state_history(config):
    print("----- CHECKPOINT -----")
    print("Values:", state.values)
    print("Next:", state.next)


English: hairs
Meaning: Strands of hair on a person’s head or body.
Hindi: 
Gujarati: 


Saved State:
----- CHECKPOINT -----
Values: {'word': 'hairs', 'meaning': 'Strands of hair on a person’s head or body.', 'hindi': '', 'gujarati': ''}
Next: ('translate_to_hindi',)
----- CHECKPOINT -----
Values: {'word': 'hairs', 'meaning': '', 'hindi': '', 'gujarati': ''}
Next: ('analyze_word',)
----- CHECKPOINT -----
Values: {}
Next: ('__start__',)


In [25]:
result = graph.invoke(
    None,
    config=config
)

print("\nEnglish:", result["word"])
print("Meaning:", result["meaning"])
print("Hindi:", result["hindi"])
print("Gujarati:", result["gujarati"])
print("\n\nSaved State:")
for state in graph.get_state_history(config):
    print("----- CHECKPOINT -----")
    print("Values:", state.values)
    print("Next:", state.next)


English: hairs
Meaning: Strands of hair on a person’s head or body.
Hindi: बाल
Gujarati: વાળ


Saved State:
----- CHECKPOINT -----
Values: {'word': 'hairs', 'meaning': 'Strands of hair on a person’s head or body.', 'hindi': 'बाल', 'gujarati': 'વાળ'}
Next: ()
----- CHECKPOINT -----
Values: {'word': 'hairs', 'meaning': 'Strands of hair on a person’s head or body.', 'hindi': 'बाल', 'gujarati': ''}
Next: ('translate_to_gujarati',)
----- CHECKPOINT -----
Values: {'word': 'hairs', 'meaning': 'Strands of hair on a person’s head or body.', 'hindi': '', 'gujarati': ''}
Next: ('translate_to_hindi',)
----- CHECKPOINT -----
Values: {'word': 'hairs', 'meaning': '', 'hindi': '', 'gujarati': ''}
Next: ('analyze_word',)
----- CHECKPOINT -----
Values: {}
Next: ('__start__',)
